# TK2 — Optimasi Model CNN CIFAR-10

Notebook ini melatih dan membandingkan tiga model:

| Model | Isi |
|---|---|
| **M1 — Baseline CNN** | 3 blok Conv + MaxPool, tanpa regularisasi |
| **M2 — CNN teroptimasi** | Batch normalization + dropout + data augmentation + hyperparameter tuning (KerasTuner Hyperband) |
| **M3 — Transfer learning** | MobileNetV2 pretrained ImageNet + fine-tuning |

**Cara menjalankan di Google Colab**

1. `Runtime → Change runtime type → T4 GPU`.
2. Upload gambar uji lewat panel **Files** di kiri — boleh langsung file-filenya, folder `test_images/`, atau `test_images.zip`.
   Format nama file: `<kelas>_<nomor>.jpg`, contoh `cat_1.jpg`. Gambar di luar 10 kelas: `other_1.jpg`.
3. `Runtime → Run all`.
4. Di akhir, `results.zip` otomatis terunduh: berisi model terbaik + semua grafik + CSV.

Sel yang perlu di-screenshot ditandai **📸 SS-xx**.

## 0. Cek GPU — 📸 SS-01

In [ ]:
!nvidia-smi

## 1. Setup

In [ ]:
!pip install -q keras-tuner tensorboard

In [ ]:
# ===== KONFIGURASI =====
QUICK = False        # True = uji cepat (data kecil, epoch sedikit) untuk cek notebook jalan
USE_DRIVE = False    # True = simpan hasil tuning ke Google Drive (aman bila sesi Colab putus)
SEED = 42
BATCH = 64

EPOCHS_M1 = 20
TUNER_MAX_EPOCHS = 15
EPOCHS_M2_FINAL = 50
EPOCHS_M3_HEAD = 10
EPOCHS_M3_FINE = 10

if QUICK:
    EPOCHS_M1, TUNER_MAX_EPOCHS, EPOCHS_M2_FINAL = 1, 2, 1
    EPOCHS_M3_HEAD, EPOCHS_M3_FINE = 1, 1

In [ ]:
import os, json, time, glob, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
import keras_tuner as kt
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
print("GPU        :", tf.config.list_physical_devices("GPU"))

RESULTS = "results"
MODEL_OUT = "model_out"
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(MODEL_OUT, exist_ok=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    TUNER_DIR = "/content/drive/MyDrive/tk2_tuner"
else:
    TUNER_DIR = "tuner"

CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

## 2. Dataset CIFAR-10

Pembagian: **train 45.000 / validasi 5.000 / test 10.000**.
Piksel dibiarkan 0–255 karena normalisasi dilakukan **di dalam model** (layer `Rescaling`),
sehingga aplikasi cukup mengirim gambar 32×32 apa adanya.

In [ ]:
# Mirror Google Cloud (tfds) jauh lebih cepat daripada server Toronto
try:
    import tensorflow_datasets as tfds
    (x_full, y_full), (x_test, y_test) = [
        tfds.as_numpy(tfds.load("cifar10", split=s, batch_size=-1, as_supervised=True, try_gcs=True))
        for s in ("train", "test")]
    print("Dataset dimuat dari mirror Google Cloud (tfds)")
except Exception as e:
    print("tfds gagal:", e, "-> unduh dari server Toronto (lambat)")
    (x_full, y_full), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_full, y_test = np.asarray(y_full).flatten(), np.asarray(y_test).flatten()

idx = np.random.permutation(len(x_full))
x_full, y_full = x_full[idx], y_full[idx]
x_train, y_train = x_full[:45000], y_full[:45000]
x_val, y_val = x_full[45000:], y_full[45000:]

if QUICK:
    x_train, y_train = x_train[:2000], y_train[:2000]
    x_val, y_val = x_val[:500], y_val[:500]
    x_test, y_test = x_test[:1000], y_test[:1000]

x_train = x_train.astype("float32"); x_val = x_val.astype("float32"); x_test = x_test.astype("float32")
print("train:", x_train.shape, " val:", x_val.shape, " test:", x_test.shape)

## 3. Fungsi bantu (plot, evaluasi, penyimpanan)

In [ ]:
comparison = []   # satu baris per model

def plot_history(hist, name, fname):
    """Kurva akurasi & loss train vs validasi."""
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist["accuracy"], label="train"); ax[0].plot(hist["val_accuracy"], label="val")
    ax[0].set_title(f"{name} — Akurasi"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(hist["loss"], label="train"); ax[1].plot(hist["val_loss"], label="val")
    ax[1].set_title(f"{name} — Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS, fname), dpi=150); plt.show()

def evaluate(model, name, train_seconds):
    """Hitung metrik test set dan simpan ke tabel perbandingan."""
    loss, acc = model.evaluate(x_test, y_test, batch_size=256, verbose=0)
    path = os.path.join(MODEL_OUT, f"{name}.keras")
    model.save(path)
    size_mb = os.path.getsize(path) / 1e6
    model.predict(x_test[:1], verbose=0)                       # pemanasan
    t0 = time.perf_counter()
    for i in range(50):
        model.predict(x_test[i:i + 1], verbose=0)
    infer_ms = (time.perf_counter() - t0) / 50 * 1000
    row = {"model": name, "test_accuracy": round(acc, 4), "test_loss": round(loss, 4),
           "params": model.count_params(), "size_mb": round(size_mb, 2),
           "train_minutes": round(train_seconds / 60, 1), "infer_ms_per_image": round(infer_ms, 1)}
    comparison.append(row)
    print(row)
    return row

AUG = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

## 4. M1 — Baseline CNN (tanpa regularisasi)

In [ ]:
def build_m1():
    inp = keras.Input((32, 32, 3))
    x = layers.Rescaling(1 / 255)(inp)
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    out = layers.Dense(10, activation="softmax")(x)
    m = keras.Model(inp, out, name="M1_baseline")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

m1 = build_m1()
m1.summary()
t0 = time.time()
h1 = m1.fit(x_train, y_train, validation_data=(x_val, y_val),
            epochs=EPOCHS_M1, batch_size=BATCH, verbose=2)
t_m1 = time.time() - t0
plot_history(h1.history, "M1 Baseline", "curve_baseline.png")
evaluate(m1, "M1_baseline", t_m1)

## 5. M2 — CNN teroptimasi + hyperparameter tuning

Teknik: **batch normalization**, **dropout**, **data augmentation**, dan **hyperparameter tuning**
dengan KerasTuner Hyperband.

| Hyperparameter | Nilai dicoba |
|---|---|
| filter blok pertama | 32, 64 |
| dropout | 0.2, 0.3, 0.4, 0.5 |
| unit Dense | 128, 256, 512 |
| learning rate | 1e-3, 5e-4, 1e-4 |
| optimizer | Adam, SGD (momentum 0.9) |

In [ ]:
def build_m2(hp):
    filters = hp.Choice("filters", [32, 64])
    dropout = hp.Choice("dropout", [0.2, 0.3, 0.4, 0.5])
    units = hp.Choice("units", [128, 256, 512])
    lr = hp.Choice("learning_rate", [1e-3, 5e-4, 1e-4])
    opt_name = hp.Choice("optimizer", ["adam", "sgd"])

    inp = keras.Input((32, 32, 3))
    x = AUG(inp)
    x = layers.Rescaling(1 / 255)(x)
    for i in range(3):
        f = filters * 2 ** i
        for _ in range(2):
            x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(dropout)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(10, activation="softmax")(x)
    m = keras.Model(inp, out, name="M2_optimized")

    opt = (keras.optimizers.Adam(lr) if opt_name == "adam"
           else keras.optimizers.SGD(lr, momentum=0.9))
    m.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

tuner = kt.Hyperband(build_m2, objective="val_accuracy", max_epochs=TUNER_MAX_EPOCHS,
                     factor=3, seed=SEED, directory=TUNER_DIR, project_name="m2",
                     overwrite=QUICK)
tuner.search_space_summary()

t0 = time.time()
tuner.search(x_train, y_train, validation_data=(x_val, y_val), batch_size=BATCH,
             callbacks=[keras.callbacks.EarlyStopping(patience=5)], verbose=2)
t_tune = time.time() - t0
print(f"Waktu tuning: {t_tune / 60:.1f} menit")

### Hasil tuning — 📸 SS-03

In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print("=== HYPERPARAMETER TERBAIK ===")
for k, v in best_hp.values.items():
    if not k.startswith("tuner/"):
        print(f"{k:15s}: {v}")

rows = []
for t in tuner.oracle.trials.values():
    if t.score is None:
        continue
    r = {k: t.hyperparameters.values.get(k) for k in ["filters", "dropout", "units", "learning_rate", "optimizer"]}
    r["epochs"] = t.hyperparameters.values.get("tuner/epochs")
    r["val_accuracy"] = round(t.score, 4)
    rows.append(r)
if not rows:
    raise RuntimeError("Semua trial gagal — cek log error tuning di atas.")
tuner_df = pd.DataFrame(rows).sort_values("val_accuracy", ascending=False)
tuner_df.to_csv(os.path.join(RESULTS, "tuner_results.csv"), index=False)
print(f"\nJumlah trial: {len(tuner_df)}  — 10 teratas:")
tuner_df.head(10)

### Latih ulang M2 dengan hyperparameter terbaik — 📸 SS-02 (model.summary)

In [ ]:
m2 = build_m2(best_hp)
m2.summary()
cb = [keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
      keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6)]
t0 = time.time()
h2 = m2.fit(x_train, y_train, validation_data=(x_val, y_val),
            epochs=EPOCHS_M2_FINAL, batch_size=BATCH, callbacks=cb, verbose=2)
t_m2 = time.time() - t0 + t_tune
plot_history(h2.history, "M2 Optimized", "curve_optimized.png")
evaluate(m2, "M2_optimized", t_m2)

## 6. M3 — Transfer learning MobileNetV2

Gambar 32×32 diperbesar ke 96×96 **di dalam model**. Fase 1: base dibekukan, hanya head yang dilatih.
Fase 2: 30 layer teratas base dibuka dan di-fine-tune dengan learning rate kecil.

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights="imagenet")
base.trainable = False

inp = keras.Input((32, 32, 3))
x = AUG(inp)
x = layers.Resizing(96, 96)(x)
x = layers.Rescaling(1 / 127.5, offset=-1)(x)       # sama dengan preprocess_input MobileNetV2
x = base(x, training=False)                        # BN base tetap mode inferensi
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(10, activation="softmax")(x)
m3 = keras.Model(inp, out, name="M3_transfer")

m3.compile(optimizer=keras.optimizers.Adam(1e-3),
           loss="sparse_categorical_crossentropy", metrics=["accuracy"])
t0 = time.time()
h3a = m3.fit(x_train, y_train, validation_data=(x_val, y_val),
             epochs=EPOCHS_M3_HEAD, batch_size=BATCH, verbose=2)

# Fase 2: fine-tune 30 layer teratas
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
m3.compile(optimizer=keras.optimizers.Adam(1e-5),
           loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h3b = m3.fit(x_train, y_train, validation_data=(x_val, y_val),
             epochs=EPOCHS_M3_FINE, batch_size=BATCH, verbose=2,
             callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])
t_m3 = time.time() - t0

h3 = {k: h3a.history[k] + h3b.history[k] for k in h3a.history}
plot_history(h3, "M3 Transfer (head + fine-tune)", "curve_transfer.png")
evaluate(m3, "M3_transfer", t_m3)

## 7. Perbandingan model — 📸 SS-04

In [ ]:
cmp_df = pd.DataFrame(comparison)
cmp_df.to_csv(os.path.join(RESULTS, "comparison.csv"), index=False)
print(cmp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(cmp_df["model"], cmp_df["test_accuracy"] * 100, color=["#9AA5B1", "#3E7CB1", "#2A9D8F"])
for i, v in enumerate(cmp_df["test_accuracy"] * 100):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center")
ax.set_ylabel("Akurasi test (%)"); ax.set_ylim(0, 100); ax.set_title("Perbandingan Akurasi Test")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, "comparison.png"), dpi=150); plt.show()

## 8. Evaluasi model terbaik: confusion matrix & classification report

In [ ]:
best_row = cmp_df.sort_values("test_accuracy", ascending=False).iloc[0]
BEST_NAME = best_row["model"]
best_model = {"M1_baseline": m1, "M2_optimized": m2, "M3_transfer": m3}[BEST_NAME]
print("Model terbaik:", BEST_NAME, "— akurasi test", best_row["test_accuracy"])

y_pred = best_model.predict(x_test, batch_size=256, verbose=0).argmax(1)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title(f"Confusion Matrix — {BEST_NAME}")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, "confusion_matrix.png"), dpi=150); plt.show()

report = classification_report(y_test, y_pred, target_names=CLASSES, digits=4, zero_division=0)
print(report)
with open(os.path.join(RESULTS, "classification_report.txt"), "w") as f:
    f.write(f"Model: {BEST_NAME}\n\n{report}")

# Pasangan kelas paling sering tertukar
pairs = [(CLASSES[i], CLASSES[j], cm[i, j]) for i in range(10) for j in range(10) if i != j]
pairs.sort(key=lambda p: -p[2])
print("Pasangan paling sering tertukar (asli -> prediksi):")
for a, b, n in pairs[:5]:
    print(f"  {a:10s} -> {b:10s}: {n}")

## 9. Pengujian dengan gambar baru (di luar dataset)

Gambar dari `test_images/` di-**crop tengah menjadi persegi** lalu di-resize ke 32×32 —
proses yang sama persis dengan aplikasi Streamlit.

In [ ]:
if os.path.exists("test_images.zip") and not os.path.isdir("test_images"):
    shutil.unpack_archive("test_images.zip", ".")

# Gambar ter-upload langsung ke /content (bukan di dalam folder)? Pindahkan otomatis.
os.makedirs("test_images", exist_ok=True)
for f in glob.glob("*.*"):
    stem = os.path.splitext(f)[0].lower()
    if (f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
            and stem.split("_")[0] in CLASSES + ["other"]):
        shutil.move(f, os.path.join("test_images", f))

def load_for_model(path):
    """Crop tengah ke persegi, resize 32x32, kembalikan array float32 (0-255)."""
    img = Image.open(path).convert("RGB")
    w, h = img.size
    s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    img = img.resize((32, 32), Image.BILINEAR)
    return np.asarray(img, dtype="float32")

files = sorted(f for f in glob.glob("test_images/**/*", recursive=True)
               if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")))
print("Jumlah gambar uji:", len(files))

models = {"M1": m1, "M2": m2, "M3": m3}
rows = []
if files:
    batch = np.stack([load_for_model(f) for f in files])
    probs = {k: m.predict(batch, verbose=0) for k, m in models.items()}
    for i, f in enumerate(files):
        label = os.path.basename(f).split("_")[0].lower()
        r = {"file": os.path.basename(f), "label": label if label in CLASSES else "(di luar kelas)"}
        for k in models:
            p = probs[k][i]
            r[f"pred_{k}"] = CLASSES[p.argmax()]
            r[f"conf_{k}"] = round(float(p.max()) * 100, 1)
            r[f"benar_{k}"] = ("Benar" if CLASSES[p.argmax()] == label else "Salah") if label in CLASSES else "-"
        rows.append(r)
    uji = pd.DataFrame(rows)
    uji.to_csv(os.path.join(RESULTS, "hasil_uji.csv"), index=False)

    known = uji[uji["label"] != "(di luar kelas)"]
    for k in models:
        if len(known):
            print(f"Akurasi gambar baru {k}: {(known[f'benar_{k}'] == 'Benar').mean() * 100:.1f}% ({len(known)} gambar)")
    display(uji)
else:
    print("Folder test_images/ kosong — upload gambar uji lalu jalankan ulang sel ini.")

In [ ]:
# Grid hasil prediksi model terbaik untuk gambar baru
if files:
    k = BEST_NAME.split("_")[0]
    n = len(files); cols = 6; rws = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rws, cols, figsize=(cols * 2.2, rws * 2.6))
    for ax in np.array(axes).flatten():
        ax.axis("off")
    for ax, r, f in zip(np.array(axes).flatten(), rows, files):
        ax.imshow(Image.open(f).convert("RGB"))
        ok = r[f"benar_{k}"]
        color = "green" if ok == "Benar" else ("red" if ok == "Salah" else "gray")
        ax.set_title(f"asli: {r['label']}\nprediksi: {r[f'pred_{k}']} ({r[f'conf_{k}']}%)",
                     fontsize=8, color=color)
    plt.suptitle(f"Prediksi gambar baru — {BEST_NAME}")
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS, "test_predictions_grid.png"), dpi=150); plt.show()

## 10. Simpan model terbaik untuk aplikasi & unduh `results.zip`

In [ ]:
best_model.save(os.path.join(MODEL_OUT, "cifar10_best.keras"))
info = {
    "model_name": BEST_NAME,
    "input_size": 32,
    "classes": CLASSES,
    "test_accuracy": float(best_row["test_accuracy"]),
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
}
with open(os.path.join(MODEL_OUT, "model_info.json"), "w") as f:
    json.dump(info, f, indent=2)
print(json.dumps(info, indent=2))

pack = "pack"
shutil.rmtree(pack, ignore_errors=True)
os.makedirs(f"{pack}/model")
shutil.copy(os.path.join(MODEL_OUT, "cifar10_best.keras"), f"{pack}/model/")
shutil.copy(os.path.join(MODEL_OUT, "model_info.json"), f"{pack}/model/")
shutil.copytree(RESULTS, f"{pack}/results")
shutil.make_archive("results", "zip", pack)
print("results.zip siap:", os.path.getsize("results.zip") / 1e6, "MB")

try:
    from google.colab import files as colab_files
    colab_files.download("results.zip")
except ImportError:
    pass